In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
dataset_csv = pd.read_csv(r"../Speculation-Map/DataLidia\Establiments_d’allotjament_turístic_inscrits_al_Registre_de_Turisme_de_Catalunya_20261116.csv",
                       index_col=0,dtype=object)

dataset_csv["Codi Municipi (IDESCAT)"]

FileNotFoundError: [Errno 2] No such file or directory: '..\\Speculation-Map\\Data\\Lidia\\Establiments_d’allotjament_turístic_inscrits_al_Registre_de_Turisme_de_Catalunya_20261116.csv'

In [4]:
conteo_municipios = (
    dataset_csv["Codi Municipi (IDESCAT)"]
    .value_counts()
    .reset_index()
)
conteo_municipios.columns = ["Codi Municipi (IDESCAT)", "NombreEdificis"]


conteo_municipios.rename(columns={"Codi Municipi (IDESCAT)": "CODIMUNI"},inplace=True)
print(conteo_municipios)

    CODIMUNI  NombreEdificis
0     080193           11420
1     439057            7204
2     171523            5401
3     430385            3804
4     170470            3690
..       ...             ...
917   252311               1
918   080879               1
919   251826               1
920   252120               1
921   250045               1

[922 rows x 2 columns]


In [5]:
dataset_csv.columns


Index(['Número inscripció', 'Dígit de control', 'Rètol', 'Estat',
       'Tipus de via', 'Nom de la via', 'Número', 'Pis', 'Porta', 'Escala',
       'Bloc', 'Codi Postal', 'Municipi', 'Codi Municipi (IDESCAT)', 'Comarca',
       'Codi Comarca (IDESCAT)', 'Província', 'Referència cadastral',
       'Cèdula d'habitabilitat', 'Grup', 'Modalitat', 'Categoria',
       'Unitat d'allotjament', 'Total places', 'Total estances', 'SSTT',
       'Marca turística', 'CIF', 'Raó Social del titular', 'Nom del titular',
       'Primer cognom', 'Segon cognom'],
      dtype='object')

In [6]:
import geopandas as gpd
import os
import sys
municipis = gpd.read_file(r"C:\Users\user\Speculation-Map\divisions-administratives-v2r1-municipis-1000000-20250730.json").to_crs(25831)
municipis.plot
municipis.columns
#gdf = gdf.merge(counts, on="CODICOMAR", how="left", validate="one_to_one")
#gdf["per_km2"] = gdf["n"] / (gdf.area / 1e6)
#ax = gdf.plot(column="per_km2", cmap="viridis", legend=True)

ModuleNotFoundError: No module named 'geopandas'

In [ ]:
municipis = municipis.merge(conteo_municipios, on="CODIMUNI", how="left", validate="one_to_one")

In [ ]:
municipis

In [ ]:

municipis.plot(column="NombreEdificis", cmap="cividis", legend=True)


In [ ]:
habitants_csv = pd.read_csv(r"C:\Users\user\Documents\aaaaa.csv",
                       index_col=0)
habitants_csv


In [ ]:
habitants1 = habitants_csv.pivot(index="MUN",columns="SEX",values="value")
habitants1["MUN_COPY"] = habitants1.index
habitants1

In [ ]:
habitants1.rename(columns={"MUN_COPY": "CODIMUNI"},inplace=True)
habitants1

In [ ]:
municipis_merged = municipis.merge(habitants1, on="CODIMUNI", how="left", validate="one_to_one")
municipis_merged

In [ ]:
municipis_merged["TOTAL"]=pd.to_numeric(municipis_merged["TOTAL"], errors="coerce")

In [ ]:
municipis_merged["fraccio"] = municipis_merged["NombreEdificis"]/municipis_merged["TOTAL"]
municipis_merged

In [ ]:
def add_scale_bar(ax, length_km=50, loc=(0.05, 0.002)):
    """Draw a scale bar of length_km on an Axes whose coordinates are in metres."""
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    x = x0 + loc[0] * (x1 - x0)
    y = y0 + loc[1] * (y1 - y0)
    ax.plot([x, x + length_km * 1000], [y, y], color="black", lw=3)
    ax.text(x + length_km * 500, y + 0.015 * (y1 - y0), f"{length_km} km", ha="center", va="bottom", fontsize=9)


def add_north_arrow(ax, loc=(0.93, 0.93)):
    ax.annotate("N", xy=loc, xytext=(loc[0], loc[1] - 0.1), xycoords="axes fraction",
                ha="center", va="center", fontsize=12, fontweight="bold",
                arrowprops={"arrowstyle": "-|>", "color": "black"})

In [ ]:
from matplotlib.colors import LogNorm
fig, axs = plt.subplots(1, 1, figsize=(11, 5))
municipis_merged.plot(column="fraccio", cmap="cividis", norm=LogNorm(),legend=True, ax=axs,
         legend_kwds={"shrink": 0.7})
axs.set_title("Habitatges turístics/habitants")
axs.set_axis_off()
add_scale_bar(axs)
add_north_arrow(axs)
plt.show()

In [ ]:
municipis_merged["quintile"] = pd.qcut(municipis_merged["fraccio"], 5, labels=["lowest 20%", "2", "3", "4", "highest 20%"])

In [ ]:
from matplotlib.colors import LogNorm
fig, axs = plt.subplots(1, 1, figsize=(11, 5))
municipis_merged.plot(column="quintile", cmap="Purples", norm=LogNorm(),legend=True, ax=axs)
axs.set_title("Habitatges turístics/habitants")
axs.set_axis_off()
axs.legend(loc='best')
add_scale_bar(axs)
add_north_arrow(axs)
plt.show()